# Crypto Lab 07 — Diffie-Hellman Key Agreement

**CYBR 3570: Applied Cryptography**  
**Topic:** Public-key key agreement, discrete logarithms, protocol authentication, and KDFs

## Today's big question

> How can two people agree on a shared secret when an eavesdropper sees every message they exchange?

Diffie-Hellman is not an encryption algorithm by itself. It is a **key agreement** technique. Its output must still be authenticated, processed through a KDF, and used with symmetric cryptography.


## Learning objectives

By the end of this notebook, you should be able to:

1. Compute a toy Diffie-Hellman exchange using modular exponentiation.
2. Explain the relationship between DLP, CDH, and DDH at a high level.
3. Demonstrate why anonymous Diffie-Hellman fails against active attackers.
4. Explain the purpose of authenticated Diffie-Hellman.
5. Explain why raw DH shared secrets should be passed through a KDF.
6. Identify unsafe parameter choices and implementation risks.
7. Add an educational DH module to your cryptographic toolkit.


## Setup

In [1]:
import hashlib
import hmac
import secrets
from dataclasses import dataclass
from typing import Optional, Tuple, List

print("Lab 07 setup complete")

Lab 07 setup complete


## Part 1 — The Diffie-Hellman idea

Diffie-Hellman uses public values and private exponents.

Public parameters:

```text
p = a large prime
g = a generator/base
```

Alice chooses private exponent `a` and publishes:

```text
A = g^a mod p
```

Bob chooses private exponent `b` and publishes:

```text
B = g^b mod p
```

Alice computes:

```text
B^a mod p = (g^b)^a mod p = g^(ab) mod p
```

Bob computes:

```text
A^b mod p = (g^a)^b mod p = g^(ab) mod p
```

Both compute the same shared secret without ever sending `a`, `b`, or `g^(ab)` directly.


In [2]:
# A tiny toy Diffie-Hellman exchange.
# These parameters are intentionally small and insecure.

p = 23
g = 5

alice_private = 6
bob_private = 15

alice_public = pow(g, alice_private, p)
bob_public = pow(g, bob_private, p)

alice_shared = pow(bob_public, alice_private, p)
bob_shared = pow(alice_public, bob_private, p)

print(f"Public parameters: p={p}, g={g}")
print(f"Alice public value A = {alice_public}")
print(f"Bob public value   B = {bob_public}")
print(f"Alice shared secret = {alice_shared}")
print(f"Bob shared secret   = {bob_shared}")
print(f"Shared secrets match? {alice_shared == bob_shared}")

Public parameters: p=23, g=5
Alice public value A = 8
Bob public value   B = 19
Alice shared secret = 2
Bob shared secret   = 2
Shared secrets match? True


### Exercise 1 — Explain the algebra

In 3–5 sentences, explain why Alice and Bob compute the same shared value even though they use different private exponents.


### Answer: The end goal is to get g^(ab). when the known value (a or b) gets used as the power of the private value (A or B) then it is the same value/equation as g^(ab). Going back to algebra this is due to the transitive property.


## Part 2 — Generators and small groups

Diffie-Hellman works inside a mathematical group. In this toy notebook, we use the multiplicative group of nonzero integers modulo a prime `p`, often written `Z_p*`.

A generator is an element whose powers produce every nonzero group element.


In [3]:
def zn_star(p: int) -> list[int]:
    """Return the nonzero elements modulo p."""
    return list(range(1, p))


def powers_mod(g: int, p: int) -> list[int]:
    """Return g^1, g^2, ..., g^(p-1) modulo p."""
    return [pow(g, k, p) for k in range(1, p)]


def is_generator(g: int, p: int) -> bool:
    """Return True if g generates all of Z_p*. Assumes p is prime."""
    return set(powers_mod(g, p)) == set(zn_star(p))

p = 13
for g in zn_star(p):
    if is_generator(g, p):
        print(f"{g} is a generator of Z_{p}*: {powers_mod(g, p)}")

2 is a generator of Z_13*: [2, 4, 8, 3, 6, 12, 11, 9, 5, 10, 7, 1]
6 is a generator of Z_13*: [6, 10, 8, 9, 2, 12, 7, 3, 5, 4, 11, 1]
7 is a generator of Z_13*: [7, 10, 5, 9, 11, 12, 6, 3, 8, 4, 2, 1]
11 is a generator of Z_13*: [11, 4, 5, 3, 7, 12, 2, 9, 8, 10, 6, 1]


### Exercise 2 — Find generators

Find all generators of `Z_17*`. For each generator, print its powers modulo 17.


In [6]:
# TODO: Find and print all generators of Z_17*.
p = 17
gen_list = []
for g in zn_star(p):
    if is_generator(g, p):
        gen_list.append(g)
print(f"{gen_list} are all generators of Z_{p}*")



[3, 5, 6, 7, 10, 11, 12, 14] are all generators of Z_17*


## Part 3 — Brute-forcing discrete logarithms

The discrete logarithm problem asks:

```text
Given g, x, and p, find a such that g^a = x mod p.
```

For tiny groups, brute force works. For cryptographic group sizes, brute force should be infeasible.


In [8]:
def discrete_log_bruteforce(g: int, x: int, p: int) -> Optional[int]:
    """Find a such that g^a = x mod p, or return None."""
    for a in range(0, p - 1):
        if pow(g, a, p) == x:
            return a
    return None

p = 23
g = 5
secret = 13
x = pow(g, secret, p)
recovered = discrete_log_bruteforce(g, x, p)

print(f"Given g={g}, p={p}, x={x}")
print(f"Recovered exponent: {recovered}")

Given g=5, p=23, x=21
Recovered exponent: 13


### Exercise 3 — Recover toy private exponents

Use `discrete_log_bruteforce()` to recover the private exponents for the following public values.


In [11]:
toy_public_values = [
    # (p, g, public_value)
    (23, 5, pow(5, 7, 23)),
    (23, 5, pow(5, 11, 23)),
    (29, 2, pow(2, 9, 29)),
]

# TODO: Recover the exponent for each public value.
for p, g, public_value in toy_public_values:
    recovered = None
    recovered = discrete_log_bruteforce(g, public_value, p)
    print(f"p={p}, g={g}, public={public_value}, recovered exponent={recovered}")

p=23, g=5, public=17, recovered exponent=7
p=23, g=5, public=22, recovered exponent=11
p=29, g=2, public=19, recovered exponent=9


## Part 4 — DLP, CDH, and DDH

Diffie-Hellman is connected to three related problems:

| Problem | Informal statement |
|---|---|
| DLP | Given `g^a`, recover `a`. |
| CDH | Given `g^a` and `g^b`, compute `g^(ab)`. |
| DDH | Given `g^a`, `g^b`, and `x`, decide whether `x = g^(ab)` or random. |

DLP is enough to break DH because recovering either private exponent lets an attacker compute the shared secret. CDH asks whether the attacker can compute the shared secret directly. DDH asks whether the attacker can distinguish the shared secret from a random group element.


### Exercise 4 — Concept check

For each scenario below, identify whether it most closely describes DLP, CDH, or DDH.


### Scenerios
- Recover Alice's private exponent from A = g^a mod p
    
    ***Your answer here*** DLP

- Compute g^(ab) after seeing g^a and g^b
    
    ***Your answer here*** CDH

- Decide whether a given value is the real shared secret or a random group element
    
    ***Your answer here*** DDH

## Part 5 — Anonymous Diffie-Hellman and man-in-the-middle attacks

Anonymous Diffie-Hellman gives Alice and Bob a shared secret against a passive eavesdropper. It does **not** authenticate either party.

An active attacker can sit between Alice and Bob and run two separate DH exchanges:

```text
Alice <---- shared secret 1 ----> Eve <---- shared secret 2 ----> Bob
```

Alice believes she shares a key with Bob. Bob believes he shares a key with Alice. In reality, both share different keys with Eve.


In [12]:
def dh_public(g: int, private: int, p: int) -> int:
    return pow(g, private, p)


def dh_shared(peer_public: int, private: int, p: int) -> int:
    return pow(peer_public, private, p)

p = 23
g = 5

# Alice and Bob's private exponents
alice_a = 6
bob_b = 15

# Eve chooses separate private exponents for each side
# Eve pretends to be Bob when talking to Alice, and Alice when talking to Bob.
eve_to_alice_d = 10
eve_to_bob_c = 4

# Public values
A = dh_public(g, alice_a, p)
B = dh_public(g, bob_b, p)
D = dh_public(g, eve_to_alice_d, p)  # sent to Alice instead of B
C = dh_public(g, eve_to_bob_c, p)    # sent to Bob instead of A

alice_thinks_shared = dh_shared(D, alice_a, p)
eve_with_alice = dh_shared(A, eve_to_alice_d, p)

bob_thinks_shared = dh_shared(C, bob_b, p)
eve_with_bob = dh_shared(B, eve_to_bob_c, p)

print("Alice's shared secret with Eve:", alice_thinks_shared, eve_with_alice)
print("Bob's shared secret with Eve:  ", bob_thinks_shared, eve_with_bob)
print("Alice and Bob share same secret?", alice_thinks_shared == bob_thinks_shared)

Alice's shared secret with Eve: 3 3
Bob's shared secret with Eve:   3 3
Alice and Bob share same secret? True


### Exercise 5 — Explain the MITM attack

Explain why the attack above works. Your answer should mention authentication.


In [13]:
exercise_5_answer = """
They are just beginning the handshake above, they have no way at the moment to verify it is going straight to the right person. this lets an attacker sit between in the key exchange to grab the key and with that any further information being exchanged in that connection.
"""
print(exercise_5_answer)


They are just beginning the handshake above, they have no way at the moment to verify it is going straight to the right person. this lets an attacker sit between in the key exchange to grab the key and with that any further information being exchanged in that connection.



## Part 6 — Authenticated Diffie-Hellman

Authenticated Diffie-Hellman adds identity verification. A common pattern is:

1. Alice sends her DH public value and a signature over that value.
2. Bob verifies the signature using Alice's known public signing key.
3. Bob sends his DH public value and a signature over that value.
4. Alice verifies Bob's signature.
5. Both compute the DH shared secret.

The important point is not the specific signature algorithm. The important point is that Eve cannot substitute her own DH public values unless she can forge signatures.


### Exercise 6 — Anonymous vs. authenticated DH

Complete the comparison table.


In [20]:
comparison = {
    "Anonymous DH protects against passive eavesdropping": "True",
    "Anonymous DH protects against active MITM": "No, when its anonymous there is no signature to verify the sender letting an attacker sit in the middle without verifying themselves",
    "Authenticated DH uses long-term identity keys": "True",
    "Authenticated DH can provide forward secrecy when ephemeral DH exponents are erased": "True",
}

for statement, answer in comparison.items():
    print(statement)
    print("  ->", answer)

Anonymous DH protects against passive eavesdropping
  -> True
Anonymous DH protects against active MITM
  -> No, when its anonymous there is no signature to verify the sender letting an attacker sit in the middle without verifying themselves
Authenticated DH uses long-term identity keys
  -> True
Authenticated DH can provide forward secrecy when ephemeral DH exponents are erased
  -> True


## Part 7 — Raw DH shared secrets are not keys

A DH shared secret is a group element, not a uniformly random byte string. Real protocols feed the DH shared secret into a KDF such as HKDF.

The KDF produces key material suitable for symmetric encryption and authentication.


In [15]:
def int_to_bytes(x: int) -> bytes:
    """Convert a positive integer to big-endian bytes."""
    if x == 0:
        return b"\x00"
    length = (x.bit_length() + 7) // 8
    return x.to_bytes(length, "big")


def hkdf_extract(salt: bytes, ikm: bytes, hashmod=hashlib.sha256) -> bytes:
    """HKDF-Extract using HMAC."""
    return hmac.new(salt, ikm, hashmod).digest()


def hkdf_expand(prk: bytes, info: bytes, length: int, hashmod=hashlib.sha256) -> bytes:
    """HKDF-Expand for length bytes."""
    output = b""
    previous = b""
    counter = 1
    while len(output) < length:
        previous = hmac.new(prk, previous + info + bytes([counter]), hashmod).digest()
        output += previous
        counter += 1
    return output[:length]


def derive_session_key(shared_secret: int, context: bytes = b"CYBR3570-DH-demo") -> bytes:
    salt = b"CYBR3570 fixed demo salt"
    ikm = int_to_bytes(shared_secret)
    prk = hkdf_extract(salt, ikm)
    return hkdf_expand(prk, context, 32)

session_key = derive_session_key(alice_shared)
print("Raw shared secret:", alice_shared)
print("Derived session key:", session_key.hex())

Raw shared secret: 2
Derived session key: 0012340b89b78c112fc72ef942bb83e0f5f65b146d45c9ef834c83f4819749d3


### Exercise 7 — Derive separate keys

Use the same shared secret to derive two different 32-byte keys:

- one with context `b"encryption"`
- one with context `b"authentication"`

Explain why deriving separate keys for separate purposes is good protocol design.


In [25]:
# TODO: Derive two separate keys from alice_shared.

encryption_key = derive_session_key(alice_shared, b"encryption")
authentication_key = derive_session_key(alice_shared, b"authentication")

print("Encryption key:    ", encryption_key.hex() if encryption_key else None)
print("Authentication key:", authentication_key.hex() if authentication_key else None)

exercise_7_answer = """
Having the same key encrypt the message and verify its you leaves you vulnerable to MITM attacks as discussed above.
"""
print(exercise_7_answer)

Encryption key:     5e1be0f3e1861fe5375fbff8440f25e1050e980f9a7970a42179c9c1875cebde
Authentication key: 62429b2d87426fac1fc2a101b4b070648ef5cf785c2b45d9e8dbf09e377ad8f3

Having the same key encrypt the message and verify its you leaves you vulnerable to MITM attacks as discussed above.



"## Part 8 — Unsafe parameters and small subgroups

Diffie-Hellman security depends on choosing safe parameters. Poor group choices can restrict possible shared secrets to a smaller set than expected or leak information about private exponents.

In real systems, do not invent your own group parameters. Use vetted standards and library defaults.


In [18]:
# Small example showing that a bad generator can produce a tiny set of values.

p = 23
for g in [1, 22, 2, 5]:
    values = sorted(set(powers_mod(g, p)))
    print(f"g={g:>2}: subgroup size={len(values):>2}, values={values}")

g= 1: subgroup size= 1, values=[1]
g=22: subgroup size= 2, values=[1, 22]
g= 2: subgroup size=11, values=[1, 2, 3, 4, 6, 8, 9, 12, 13, 16, 18]
g= 5: subgroup size=22, values=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]


### Exercise 8 — Parameter review

For the values printed above, which choices of `g` obviously look unsafe? Why?


In [19]:
exercise_8_answer = """
1 and 22, both of them have an extremely small amount of values available, making it vulnerable to brute force.
"""
print(exercise_8_answer)


1 and 22, both of them have an extremely small amount of values available, making it vulnerable to brute force.



## Part 9 — Toolkit integration

Add the following module to your toolkit:

```text
crypto_toolkit/
    key_agreement/
        __init__.py
        diffie_hellman.py
```

Suggested functions:

```python
dh_public(g, private, p)
dh_shared(peer_public, private, p)
discrete_log_bruteforce(g, x, p)
derive_session_key(shared_secret, context)
is_generator(g, p)
```

Remember: this module is educational. Real projects should use established libraries and standardized groups.


## Engineering checkpoint

Answer the following questions in the dictionary below.


In [26]:
engineering_checkpoint = {
    "Why is Diffie-Hellman called key agreement rather than encryption?": "It is rarely used for encryption due to the ",
    "Why does anonymous DH fail against active attackers?": "It has no way to clearly verify you are speaking to the relevant person leaving it vulnerable to MITM attacks",
    "What does forward secrecy protect?": "It protects past communication sessions and recorded encrypted traffic from being decrypted later",
    "Why should protocols use a KDF after DH?": "So they can sign it verifying its coming from them, lowering the risk of a MITM attack",
    "What should a security review check before accepting DH parameters?": "The signature",
}

for question, answer in engineering_checkpoint.items():
    print(question)
    print(answer)
    print()

Why is Diffie-Hellman called key agreement rather than encryption?
It is rarely used for encryption due to the 

Why does anonymous DH fail against active attackers?
It has no way to clearly verify you are speaking to the relevant person leaving it vulnerable to MITM attacks

What does forward secrecy protect?
It protects past communication sessions and recorded encrypted traffic from being decrypted later

Why should protocols use a KDF after DH?
So they can sign it verifying its coming from them, lowering the risk of a MITM attack

What should a security review check before accepting DH parameters?
The signature



## Submission checklist

Before submitting, make sure you have:

- [ ] Completed all written responses.
- [ ] Implemented the missing code exercises.
- [ ] Run the notebook from top to bottom without errors.
- [ ] Added this week's functions to your cryptographic toolkit.
- [ ] Updated your engineering log.

## Final reflection

Return to today's big question:

> How can two people agree on a shared secret when an eavesdropper sees every message they exchange?

Explain the answer while also identifying what Diffie-Hellman does **not** solve by itself.


In [27]:
final_reflection = """
They do it by using a Diffie-Hellman key exchange, this uses one way math functions to agree on a secret. this lets them establish communication and exchange keys before an attacker can intervene. But this method could be subjugated if an attacker can intercept the initial exchange and do a MITM attack. But with a correctly implemented signature on the exchange this can be mitigated.
"""
print(final_reflection)


They do it by using a Diffie-Hellman key exchange, this uses one way math functions to agree on a secret. this lets them establish communication and exchange keys before an attacker can intervene. But this method could be subjugated if an attacker can intercept the initial exchange and do a MITM attack. But with a correctly implemented signature on the exchange this can be mitigated. 

